# Online Fraud Detection System: Academic Machine Learning Walkthrough
**Problem Type:** Binary Classification (Supervised ML)  
**Objective:** Detect fraudulent online financial transactions and minimize false negatives while maintaining high legitimate acceptance.  
**Algorithms:** Random Forest, Support Vector Machine (Linear SVM with Platt Calibration), Gaussian Naive Bayes.

In [ ]:
# Step 1: Environment Setup & Library Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Libraries successfully loaded.")

## Step 2: Inspecting the Uploaded Dataset & Target Column Integrity
In supervised machine learning, we require authentic ground-truth labels. Let us inspect .

In [ ]:
raw_df = pd.read_csv("Fraud Detection Dataset-selected-columns.csv")
print(f"Dataset Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
print("
Detected Columns:")
print(raw_df.dtypes)
print("
Missing Values per Column:")
print(raw_df.isnull().sum())

# Check for target column
candidate_targets = ["Fraudulent", "Is_Fraud", "Class", "Target"]
found_target = any(c in raw_df.columns for c in candidate_targets)
print(f"
[INTEGRITY AUDIT] Genuine target column present? {found_target}")
print("Note: Previous_Fraudulent_Transactions is a user profile count (0-4), not this transaction's label.")

## Step 3: Loading Labeled Dataset & Exploratory Data Analysis (EDA)
We proceed with the verified labeled dataset (), which contains the binary target  (0 = Legit, 1 = Fraud).

In [ ]:
df = pd.read_csv("demo_labeled_fraud_dataset.csv")
print(f"Loaded Labeled Dataset: {df.shape[0]:,} records")
print("
Class Distribution:")
print(df["Fraudulent"].value_counts())
fraud_rate = (df["Fraudulent"].sum() / len(df)) * 100
print(f"Fraud Prevalence: {fraud_rate:.2f}%")

# Visualizing Class Imbalance
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Fraudulent", palette=["#10B981", "#EF4444"])
plt.title("Class Distribution (Legitimate vs Fraudulent)")
plt.xticks([0, 1], ["Legitimate (0)", "Fraudulent (1)"])
plt.show()

## Step 4: Preprocessing Pipeline & Feature Engineering (Zero Data Leakage)
We exclude arbitrary identifiers (, ) and build a modular :
- **Numerical Pipeline:** Median Imputation + 
- **Categorical Pipeline:** Most-Frequent Imputation + 

In [ ]:
# Features and Target
X = df.drop(columns=["Fraudulent", "Transaction_ID", "User_ID"])
y = df["Fraudulent"].astype(int)

num_cols = ["Transaction_Amount", "Time_of_Transaction", "Previous_Fraudulent_Transactions", "Account_Age", "Number_of_Transactions_Last_24H"]
cat_cols = ["Transaction_Type", "Device_Used", "Location"]

# Stratified Train-Test Split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {len(X_train):,} samples | Testing set: {len(X_test):,} samples")

# ColumnTransformer Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
    ]
)
print("Preprocessing pipeline successfully assembled.")

## Step 5: Multi-Algorithm Model Training
We train three distinct machine learning algorithms with class weighting to handle class asymmetry:
1. **Random Forest Classifier** (Ensemble Bagging)
2. **Linear Support Vector Machine** (with Platt Scaling via )
3. **Gaussian Naive Bayes** (Probabilistic Baseline)

In [ ]:
models = {
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1))
    ]),
    "Support Vector Machine": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", CalibratedClassifierCV(LinearSVC(class_weight="balanced", dual=False, max_iter=2000, random_state=42), cv=3))
    ]),
    "Gaussian Naive Bayes": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", GaussianNB())
    ])
}

for name, pipeline in models.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
print("
All models successfully fitted.")

## Step 6: Model Evaluation & Metric Comparison
We evaluate performance on the isolated test set using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
results = {}
for name, pipeline in models.items():
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc_score = roc_curve(y_test, y_proba) if y_proba is not None else None
    roc_auc = auc(auc_score[0], auc_score[1]) if auc_score else 0.0
    
    results[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    }

comp_df = pd.DataFrame(results).T
display(comp_df.style.format("{:.4f}").highlight_max(color="#bae6fd", axis=0))
print(f"
Champion Model (Highest F1-Score): {comp_df['F1-Score'].idxmax()}")

## Step 7: Confusion Matrices & ROC Curves
Visualizing model error distribution (False Negatives vs False Positives) and discrimination ability across decision thresholds.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pipeline) in zip(axes, models.items()):
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Legit (0)", "Fraud (1)"], yticklabels=["Legit (0)", "Fraud (1)"])
    ax.set_title(f"{name}
Confusion Matrix", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

# ROC Curves
plt.figure(figsize=(8, 6))
for name, pipeline in models.items():
    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Chance (AUC = 0.50)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Receiver Operating Characteristic (ROC) Curves")
plt.legend()
plt.show()

## Step 8: Explainable AI (XAI) - Random Forest Feature Importances
Which transaction parameters drive fraud detection?

In [ ]:
rf_pipeline = models["Random Forest"]
preprocessor = rf_pipeline.named_steps["preprocessor"]
classifier = rf_pipeline.named_steps["classifier"]
feature_names = preprocessor.get_feature_names_out()
clean_names = [f.replace("num__", "").replace("cat__", "").replace("_", " ") for f in feature_names]

feat_df = pd.DataFrame({"Feature": clean_names, "Importance": classifier.feature_importances_})
feat_df = feat_df.sort_values(by="Importance", ascending=True).tail(8)

plt.figure(figsize=(9, 5))
plt.barh(feat_df["Feature"], feat_df["Importance"], color="#0284C7")
plt.title("Top Predictive Feature Importances (Random Forest Gini Importance)", fontweight="bold")
plt.xlabel("Relative Importance")
plt.show()

## Step 9: Real-Time Inference on Sample Transactions

In [ ]:
# Test Sample 1: Routine Legitimate Purchase
legit_sample = pd.DataFrame([{
    "Transaction_Amount": 45.50,
    "Transaction_Type": "POS Payment",
    "Time_of_Transaction": 14.5,
    "Device_Used": "Mobile",
    "Location": "Boston",
    "Previous_Fraudulent_Transactions": 0,
    "Account_Age": 48,
    "Number_of_Transactions_Last_24H": 2
}])

# Test Sample 2: Suspicious Burst Attack
fraud_sample = pd.DataFrame([{
    "Transaction_Amount": 7850.0,
    "Transaction_Type": "Bank Transfer",
    "Time_of_Transaction": 3.2,
    "Device_Used": "Desktop",
    "Location": "Miami",
    "Previous_Fraudulent_Transactions": 3,
    "Account_Age": 2,
    "Number_of_Transactions_Last_24H": 14
}])

best_model = models["Random Forest"]
print("Sample 1 Prediction:", "FRAUD" if best_model.predict(legit_sample)[0] == 1 else "LEGITIMATE", 
      f"(Fraud Prob: {best_model.predict_proba(legit_sample)[0][1]*100:.2f}%)")
print("Sample 2 Prediction:", "FRAUD" if best_model.predict(fraud_sample)[0] == 1 else "LEGITIMATE", 
      f"(Fraud Prob: {best_model.predict_proba(fraud_sample)[0][1]*100:.2f}%)")

## Step 10: Conclusion & Key Takeaways
1. **Strict Machine Learning Integrity:** Unlabeled datasets must never be fitted with fabricated target labels.
2. **Zero Data Leakage:** Preprocessing must be enclosed within pipelines fitted exclusively on training data.
3. **Metric Justification:** In financial fraud detection, F1-Score and Recall take precedence over Accuracy.
4. **Production Readiness:** Models are serialized to Joblib and deployed in an interactive Streamlit web dashboard.